# ColabFold MSA Search

This notebook demonstrates `run_colabfold_search`, which generates multiple sequence alignments (MSAs) for protein sequences by searching large homolog databases using MMseqs2. The tool supports both a remote API mode (no local setup required) and a local database mode for higher throughput. The resulting MSAs can be used directly as input to structure prediction tools such as AlphaFold2.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("colabfold_search")
display_overview("colabfold_search")
display_docs_section("colabfold_search", "Background")

# ColabFold Search

> ColabFold MSA Search generates [Multiple Sequence Alignments](https://en.wikipedia.org/wiki/Multiple_sequence_alignment) (MSAs) for protein sequences by searching large sequence databases for homologs. MSAs are a foundational input for structure prediction (AlphaFold, ESMFold), co-evolutionary analysis, and conservation scoring. This tool wraps [MMSeqs2](https://mmseqs.com/) for fast local search and the ColabFold API for remote search.

## Background

Multiple Sequence Alignments capture the evolutionary history of a protein family by aligning [homologous](https://en.wikipedia.org/wiki/Homology_\(biology\)) sequences found across organisms. Each column in the alignment represents a structural position, and the patterns of conservation and covariation encode information about:

* **Structural constraints**: Positions buried in the protein core are highly conserved
* **Functional residues**: Active site and binding site residues show conservation
* **[Coevolution](https://en.wikipedia.org/wiki/Coevolution)**: Residue pairs that co-vary indicate spatial contacts, which is the key signal AlphaFold2 uses for structure prediction
* **Evolutionary rate**: The depth (number of homologs) of the MSA correlates with prediction confidence

ColabFold uses MMSeqs2 (Many-against-Many sequence searching) for fast homology detection. MMSeqs2 is \~100x faster than BLAST with comparable sensitivity, making it practical to search databases with billions of sequences.

## Available tools

In [2]:
display_available_tools("colabfold_search")

- **`run_colabfold_search()`** — Generate Multiple Sequence Alignments using ColabFold local database search

### `run_colabfold_search`

`run_colabfold_search` searches protein sequences against UniRef and optionally metagenomic databases to produce multiple sequence alignments. Each query returns an MSA containing the input sequence plus all identified homologs, with gap characters inserted to maintain positional correspondence. The remote mode calls the ColabFold API and requires no local database setup, while local mode runs MMseqs2 directly against a downloaded database for higher throughput and no rate limiting. Results are returned as MSA objects that expose alignment statistics and can be exported in A3M or FASTA format for use in downstream structure prediction pipelines.

In [3]:
from pathlib import Path

from proto_tools import (
    ColabfoldSearchConfig,
    ColabfoldSearchInput,
    run_colabfold_search,
)

In [4]:
# Display input docs
display_api_reference("colabfold_search", "input", "run_colabfold_search")

# Hemoglobin subunit alpha (human) -- first 60 residues
sequences = ["MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG"]
inputs = ColabfoldSearchInput(queries=sequences)

**Input** — `ColabfoldSearchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `queries` | `List[ColabfoldSearchQuery]` | required | List of search queries. Each query contains a protein sequence and optional identifier. After validation, always a list of ColabfoldSearchQuery instances regardless of input format. |
| `sequence` | `string` | required | Protein sequence to search for homologs. Must be a non-empty string containing amino acid characters. |
| `sequence_id` | `string` |  | Optional identifier for this sequence. Used for output file naming and result tracking. If not provided, will be auto-generated as seq\_0, seq\_1, etc. |

In [5]:
# Display config docs
display_api_reference("colabfold_search", "config", "run_colabfold_search")

config = ColabfoldSearchConfig(
    search_mode="remote",  # Use the ColabFold API (no local setup needed)
    use_metagenomic_db=False,
)

**Config** — `ColabfoldSearchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `use_metagenomic_db` | `boolean` | `False` | Whether to include metagenomic/environmental databases in the search. Metagenomic databases can improve MSA quality for some sequences but increase search time. Supported in both local and remote modes. |
| `msa_db_dir` | `string` | `/large_storage/hielab/brk/databases/colabfold` | Only used if search\_mode is "local". Path to the local ColabFold/MMSeqs2 database directory. This directory should contain the database files downloaded using the setup\_databases.sh script. To download databases, run: ./setup\_databases.sh |
| `database_name` | `string` | `uniref30_2302_db` | Only used if search\_mode is "local". Name of the database to use. If not provided, the tool will automatically detect the available databases and use one. |
| `sensitivity` | `number` |  | Only used if search\_mode is "local". MMseqs2 sensitivity parameter (1.0-9.0). Higher values increase sensitivity and may find more remote homologs, but significantly slow down the search. Lower values are faster but may miss distant homologs. If None, uses the default sensitivity (\~8). Default: None. |
| `use_gpu` | `boolean` | `False` | Only used if search\_mode is "local". Whether to enable GPU-accelerated search using MMseqs2-GPU. Requires a GPU-capable MMseqs2 binary and NVIDIA GPU (Turing gen or newer). Only available on Linux. Raises ValueError if set with search\_mode="remote" or on unsupported platforms. When enabled, passes --gpu 1 to colabfold\_search. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `search_mode` | `enum` | `remote` | Mode to use for MSA search. Available options: `local`, `remote` |
| `output_dir` | `string` |  | Directory where output MSA files will be saved. A subdirectory named 'msas' will be created to store A3M format alignment files. Each sequence will get its own A3M file named by its sequence ID. If None, uses the default cache directory (\$PROTO\_HOME/colabfold\_search). Default: None. |
| `num_threads` | `integer` |  | Only used if search\_mode is "local". Number of CPU threads to use for parallel processing. If None, automatically detects and uses all available CPU cores. Must be at least 1 if specified. Default: None (auto-detect). |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `3600` | Maximum execution time in seconds. Full database searches can take more than 10 minutes. Default: 3600. |

In [6]:
# Run the search
result = run_colabfold_search(inputs, config)

Running run_colabfold_search [00:00]

In [7]:
# Display output docs
display_api_reference("colabfold_search", "output", "run_colabfold_search")

print(f"Found {result.results[0].num_homologs_found} homologs.")

**Output** — `ColabfoldSearchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `List[ColabfoldSearchResult]` | required | List of search results, one per input query. Each result contains the path to the generated A3M file and metadata. The order matches the input queries order. |
| `msa` | `MSA` | required | The Multiple Sequence Alignment object containing the homologous sequences. None if no homologs were found (only the query sequence would be present). |
| `sequence_id` | `string` | required | Identifier for the sequence that was searched. |

Found 3677 homologs.


## Batch Search

Searching multiple sequences at once is more efficient than making separate calls. You can optionally provide custom sequence IDs using tuple format for easier tracking of results. Each query is searched independently and produces its own MSA in the output.

In [8]:
queries = [
    ("MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG", "hemoglobin_alpha"),
    ("MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPK", "hemoglobin_beta"),
]

inputs = ColabfoldSearchInput(queries=queries)

config = ColabfoldSearchConfig(
    search_mode="remote",
    use_metagenomic_db=False,
)

batch_result = run_colabfold_search(inputs, config)

for res in batch_result.results:
    print(f"{res.sequence_id}: {res.num_homologs_found} homologs found")

Running run_colabfold_search [00:00]

hemoglobin_alpha: 3677 homologs found
hemoglobin_beta: 3956 homologs found


## Inspecting MSA Results

Each result contains an MSA object with properties for exploring the alignment. The number of sequences includes the query itself, so the homolog count is one less than the total. The alignment length equals the length of the query sequence since ColabFold aligns all hits to the query coordinates. The average gap fraction indicates how divergent the homologs are from the query.

In [9]:
msa = batch_result.results[0].msa

print(f"Number of sequences: {msa.num_sequences}")
print(f"Alignment length:    {msa.alignment_length}")
print(f"Average gap fraction: {msa.average_gap_fraction:.3f}")
print(f"\nQuery sequence (first entry):")
print(f"  ID:  {msa.sequence_ids[0]}")
print(f"  Seq: {msa[0][:60]}...")

Number of sequences: 3678
Alignment length:    60
Average gap fraction: 0.063

Query sequence (first entry):
  ID:  101
  Seq: MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG...


## Export Results

MSA results can be exported to standard alignment formats for use in downstream pipelines. A3M format is a compressed alignment format used by ColabFold and AlphaFold2 as direct input to structure prediction. FASTA format is a widely compatible alternative suitable for general-purpose sequence analysis tools.

In [10]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to A3M files (one per sequence)
batch_result.export("colabfold_a3m", export_path=output_dir, file_format="a3m")
print("A3M files exported to ./example_output/colabfold_a3m/")

# Export to FASTA format
batch_result.export("colabfold_fasta", export_path=output_dir, file_format="fasta")
print("FASTA files exported to ./example_output/colabfold_fasta/")

A3M files exported to ./example_output/colabfold_a3m/
FASTA files exported to ./example_output/colabfold_fasta/
